#Initialization

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

#Read silver table

In [0]:
df = spark.table("pcat.silver.customers")
df = df.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")

#Writing Gold Table

In [0]:
df.write \
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable("pcat.gold.sb_dim_customers")

#Merging Data source with parent table

In [0]:
delta_table = DeltaTable.forName(spark, "pcat.gold.dim_customers")
df_child_customers = spark.table("pcat.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()